# Submit the DNA full-cell simulation on the Delta Gateway

The notebook copies the workshop template to your bgvl folder and launches the ~14 hour GPU simulation.

- **Allocation:** `bgvl-delta-gpu`, **4DCell (LAMMPS/LM)** container, **1 GPU**
- **Kernel:** default Python (setup cells) — LM 2.5 is not required here
- **Background reading:** [`README.md`](README.md) sections 6–12
- **VMD:** README section 12 (Open OnDemand Desktop)

**How to run:** click **Run → Run All Cells**, then watch section 5 to track progress.

---
## 1. Setting paths and log directory

Each Gateway login gets its own Jupyter session and GPU. Your personal workspace on bgvl is:

```
/projects/bgvl/$USER/DNA_SummerSchool_2026/
├── scripts/
├── data/          ← simulation output (trajectory, coords, …)
└── logs/          ← nohup log for this run
```

Shared read-only workshop files (template source, Apptainer image) live under `/projects/bgvl/SummerSchool_2026/DNA/files/`.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

USER = os.environ["USER"]
HOME = Path("/home/user/workspace")
REPO = HOME / "SummerSchool_2026" / "DNA"
BGVL_DNA = Path("/projects/bgvl/SummerSchool_2026/DNA")
PRELAUNCH = BGVL_DNA / "files" / "prelaunch_dna_workshop.sh"
WORK_ROOT = Path(f"/projects/bgvl/{USER}")
SIM_ROOT = WORK_ROOT / "DNA_SummerSchool_2026"
SCRIPTS = SIM_ROOT / "scripts"
LOGDIR = SIM_ROOT / "logs"
LOGDIR.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = "/tmp"
os.environ["HOME"] = str(HOME)
conda_bin = "/opt/conda/envs/lm_2.5_dev/bin"
if Path(conda_bin).is_dir():
    os.environ["PATH"] = conda_bin + ":" + os.environ.get("PATH", "")

if not REPO.is_dir() and not PRELAUNCH.is_file():
    subprocess.run(
        ["git", "clone", "https://github.com/Luthey-Schulten-Lab/SummerSchool_2026.git"],
        cwd=HOME,
        check=True,
    )

print("Your NCSA username:", USER)
print("Your personal sim directory:", SIM_ROOT)
print("Log directory:", LOGDIR)
print("Can write bgvl:", os.access(WORK_ROOT.parent, os.W_OK))

---
### Environment check (optional)

The Gateway **4DCell** image should provide `btree_chromo` and `gen_sc_chain` under **`/Software/`**.

In [ ]:
import sys

print("=" * 60)
print("Environment check")
print("=" * 60)
print(f"\nPython: {sys.executable}")
print(f"Version: {sys.version.split()[0]}")

BTREE = "/Software/btree_chromo/build/apps/btree_chromo"
SCCHAIN = "/Software/sc_chain_generation/src/gen_sc_chain"
SIF = "/projects/bgvl/SummerSchool_2026/DNA/files/DNA_summer2025.sif"

print("\n--- Executables ---")
print(f"btree_chromo:  {'OK' if Path(BTREE).is_file() else 'NOT FOUND'}  ({BTREE})")
print(f"gen_sc_chain:   {'OK' if Path(SCCHAIN).is_file() else 'NOT FOUND'}  ({SCCHAIN})")
print(f"Apptainer SIF:  {'OK' if Path(SIF).is_file() else 'NOT FOUND'}  ({SIF})")
print(f"apptainer:      {shutil.which('apptainer') or shutil.which('singularity') or 'NOT FOUND'}")
print("\n" + "=" * 60)

---
## 2. Copy the workshop template to your bgvl directory

Run this cell **once in your own Gateway session**. It copies the read-only template from the shared workshop folder into **your** directory only:

```
/projects/bgvl/$USER/DNA_SummerSchool_2026/
```

Other participants have separate folders under `/projects/bgvl/<their-username>/` and do not share your simulation output.

In [ ]:
prelaunch = REPO / "files" / "prelaunch_dna_workshop.sh"
if not prelaunch.is_file():
    prelaunch = PRELAUNCH
if not prelaunch.is_file():
    raise FileNotFoundError(f"prelaunch script not found: {prelaunch}")

subprocess.run(["bash", str(prelaunch)], check=True)
print(f"Copied template for user {USER} → {WORK_ROOT}")
print("Scripts:", list(SCRIPTS.glob("*")))

---
## 3. Create a log file

Pick a unique log name for each run (do not overwrite a previous run's log).

In [ ]:
RUN_LOG = LOGDIR / "dna_run.log"
print("RUN_LOG:", RUN_LOG)

---
## 4. Start the simulation

Launches **`run_sc_chain_generation.sh`** then **`run_btree_chromo.py`** (91 biological minutes, ~14 h on an A100) in the **background** with `nohup`.

**Important:**
- Use a **new log filename** in the cell below if you already ran once.
- Do not start a second copy while one is still running (cell 6 lists PIDs).
- The simulation writes output under `DNA_SummerSchool_2026/data/` including `summerschool.lammpstrj` when finished.

In [ ]:
_sim_root = str(SIM_ROOT)
_run_log = str(RUN_LOG)
_user = os.environ["USER"]
print("Will run in:", _sim_root)
print("Log file:", _run_log)

In [ ]:
%%bash -s "{_sim_root}" "{_run_log}" "{_user}"
set -e
SIM_ROOT="$1"
RUN_LOG="$2"
USER="$3"
SCRIPTS="${SIM_ROOT}/scripts"
SIF="/projects/bgvl/SummerSchool_2026/DNA/files/DNA_summer2025.sif"
BTREE="/Software/btree_chromo/build/apps/btree_chromo"

mkdir -p "${SIM_ROOT}/logs" "${SIM_ROOT}/data/coords" "${SIM_ROOT}/data/loops" "${SIM_ROOT}/data/rep_states"

if pgrep -f "run_btree_chromo.py" > /dev/null; then
  echo "ERROR: a DNA simulation is already running:"
  pgrep -af "run_btree_chromo.py" || true
  exit 1
fi

export TMPDIR=/tmp
export HOME=/home/user/workspace
export PATH="/opt/conda/envs/lm_2.5_dev/bin:${PATH}"
export LD_LIBRARY_PATH="/usr/local/lib64:/usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/Software/LAMMPS/OMP_GPU_Kokkos/lib"
export CUDA_VISIBLE_DEVICES=0

run_native() {
  cd "${SCRIPTS}"
  nohup bash -c 'bash run_sc_chain_generation.sh && python3 run_btree_chromo.py' \
    > "${RUN_LOG}" 2>&1 &
  echo "Simulation started (native /Software), background PID: $!"
  echo "Log: ${RUN_LOG}"
}

run_apptainer() {
  nohup apptainer run --nv --writable-tmpfs --no-home --containall \
    --bind "${SIM_ROOT}:/mnt" "${SIF}" /bin/bash -c \
    "export LD_LIBRARY_PATH=/usr/local/lib64:/usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/.singularity.d/libs && cd /mnt/scripts && bash run_sc_chain_generation.sh && python3 run_btree_chromo.py" \
    > "${RUN_LOG}" 2>&1 &
  echo "Simulation started (Apptainer), background PID: $!"
  echo "Log: ${RUN_LOG}"
}

if [ -x "${BTREE}" ]; then
  run_native
elif [ -f "${SIF}" ] && command -v apptainer >/dev/null; then
  run_apptainer
else
  echo "ERROR: neither native btree_chromo nor Apptainer SIF is available."
  exit 1
fi

---
## 5. Track progress

Re-run the cells below while the simulation runs (~14 hours). The last lines of the log show the current timestep.

In [ ]:
if RUN_LOG.is_file():
    subprocess.run(["tail", "-n", "80", str(RUN_LOG)], check=False)
else:
    print(f"No log yet: {RUN_LOG}")

In [ ]:
traj = SIM_ROOT / "data" / "summerschool.lammpstrj"
print("Trajectory ready:", traj.is_file())
if traj.is_file():
    print(f"Size: {traj.stat().st_size / 1e6:.1f} MB")
print(traj)

---
## 6. Cancel the simulation (optional)

Use this if you started a run by mistake. Copy the PID from the listing into **`JOB_PID`** in the next cell.

In [ ]:
%%bash
echo "Running DNA jobs:"
pgrep -af "run_btree_chromo.py" || echo "(none)"

In [ ]:
JOB_PID = None  # e.g. 148 — set to the integer PID from the cell above

if not JOB_PID:
    print("Set JOB_PID to an integer, then re-run this cell.")
else:
    r = subprocess.run(["kill", str(JOB_PID)], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"Sent SIGTERM to PID {JOB_PID}.")
    else:
        print(f"kill failed: {r.stderr or r.stdout or 'no such process'}")